Продолжение рассмотрения инженерного алгоритма, используемого в численном  
		электродинамическом расчёте волнового двигателя. 
		Основная тестовая и расчётная  среда — файл MenDrive_real(3).ipynb.	
		Исследование будет проводиться в несколько этапов. 
		Этап 1.
		Проанализируй критически файл с выводом дисперсионного уравнения 
		MenDrive_wave_equation.ipynb
		найди ошибки.

Давайте внимательно разберу весь notebook.Теперь у меня есть полная картина. Вот подробный критический анализ `MenDrive_wave_equation.ipynb`.

---

## Критический анализ MenDrive_wave_equation.ipynb

### Ошибка 1. Неверные формулы квантования k_y в markdown (Cell 1)

В описании граничных условий написано:

> **TM:** $E_z \propto \cos(k_y y)$, $k_y = \frac{m\pi}{2b}$, $m=1,2,\dots$
>
> **TE:** $H_z \propto \sin(k_y y)$, $k_y = \frac{m\pi}{2b}$, $m=0,1,2,\dots$

Обе формулы **неверны**. Правильные условия:

**Для TM** ($E_z = 0$ при $y = \pm b$):
- Если $E_z \propto \cos(k_y y)$: $\cos(k_y b) = 0$ → $k_y = \frac{(2m+1)\pi}{2b}$, не $\frac{m\pi}{2b}$.
- Если $E_z \propto \sin(k_y y)$: $\sin(k_y b) = 0$ → $k_y = \frac{m\pi}{b}$, $m = 1,2,\dots$

**Для TE** ($\partial H_z/\partial y = 0$ при $y = \pm b$):
- Если $H_z \propto \sin(k_y y)$: $\cos(k_y b) = 0$ → $k_y = \frac{(2m+1)\pi}{2b}$
- Если $H_z \propto \cos(k_y y)$: $\sin(k_y b) = 0$ → $k_y = \frac{m\pi}{b}$, $m = 0,1,2,\dots$

Формула $m\pi/(2b)$ порождает значения, для которых граничные условия **не выполняются** (например, при $m=2$: $\cos(\pi) = -1 \neq 0$).

### Ошибка 2. Несогласованность кода с markdown (Cells 2–4)

В коде Cell 2 для TE-моды:
```python
ky_TE_sin = (2*m+1)*pi/(2*b)   # ← правильно
ky_TE_cos = (2*m+1)*pi/b        # ← НЕВЕРНО
```

Для $H_z = \cos(k_y y)$ граничное условие $\partial H_z/\partial y = -k_y\sin(k_y y) = 0$ при $y=\pm b$ требует $\sin(k_y b)=0$, то есть $k_y = m\pi/b$ для **всех** $m = 0,1,2,\dots$ Формула $(2m+1)\pi/b$ даёт **только нечётные** кратные $\pi/b$, пропуская $m=0$ (основную моду) и все чётные.

### Ошибка 3. Представление поля через $e^{ik_y y}$ вместо $\sin/\cos$ (Cell 6)

```python
phase_y = exp(I * (k_y * y))
phase = exp(I * (k_z * z - omega * t)) * phase_y
E_vec = Eu.vector_field(E_x * phase, E_y * phase, E_z * phase)
```

Поля представлены с y-зависимостью в виде бегущей волны $e^{ik_y y}$, а не стоячей волны $\sin(k_y y)$ или $\cos(k_y y)$. Для **вывода дисперсионного уравнения** это допустимо — дисперсионное соотношение одинаково для бегущих и стоячих волн. Однако это создаёт **внутреннюю несогласованность** с обсуждением граничных условий в Cell 1, где явно вводятся $\sin/\cos$. Более того, при переходе к расчёту конкретных полей (в MenDrive_real_3_) это различие становится принципиальным.

### Ошибка 4 (потенциально серьёзная). Некорректная «коррекция» в двойном роторе (Cells 39, 42)

```python
# Cell 39
rot_B = curl(B_vec) - mu_xx*(curl(H_vec) - rhs_eq_rot_H)
# Cell 42
rot_D = curl(D_vec) - epsilon_xx*(curl(E_vec) - rhs_eq_rot_E)
```

Корректирующий член `mu_xx*(curl(H_vec) - rhs_eq_rot_H)` **тождественно равен нулю**, если уравнения Максвелла выполняются точно (а они выполняются — это символьные, а не численные выражения). Поэтому `rot_B = curl(B_vec)` и коррекция избыточна.

Но **концептуальная ошибка** в том, что используется скалярный множитель `mu_xx` вместо тензорного. Для **анизотропной** среды с $\mu_{xx} \neq \mu_{yy} \neq \mu_{zz}$ тождество $\nabla\times(\hat{\mu}\cdot\vec{H}) = \mu_{xx}\cdot\nabla\times\vec{H} + \text{correction}$ **неверно**. Правильная связь:

$$[\nabla\times(\hat{\mu}\cdot\vec{H})]_z = \mu_{yy}\frac{\partial H_y}{\partial x} - \mu_{xx}\frac{\partial H_x}{\partial y}$$

что не равно $\mu_{xx}\cdot[\nabla\times\vec{H}]_z = \mu_{xx}\left(\frac{\partial H_y}{\partial x} - \frac{\partial H_x}{\partial y}\right)$.

Сейчас эта ошибка «скрыта», поскольку корректирующий член обнуляется. Но если в будущем код будет модифицирован (например, для численных методов с приближёнными полями), ошибка проявится.

### Ошибка 5 (принципиальная). Отсутствие явной установки $E_z=0$ / $H_z=0$ при разделении мод (Cells 52–64)

Для **TE-моды** код берёт z-компоненту уравнения $\nabla\times\nabla\times\vec{H}$ (Cell 48), подставляет выражения для поперечных компонент через $E_z$ и $H_z$ (Cell 52), и затем извлекает:

```python
eq_TE_k_Hz = eq_TE.lhs().coefficient(H_z)         # коэффициент при H_z
eq_TE_k_d2Hz_dx2 = eq_TE.lhs().coefficient(d²H_z/dx²)  # коэффициент при H_z''
K2_x_TE = eq_TE_k_Hz / eq_TE_k_d2Hz_dx2
```

Но **нигде не ставится** $E_z = 0$. Для общей анизотропной среды с диагональными $\hat{\varepsilon}$, $\hat{\mu}$ уравнение двойного ротора **связывает** $E_z$ и $H_z$ — чистых TE/TM мод не существует. Извлекая только коэффициенты при $H_z$ и $H_z''$, код **молча отбрасывает** возможные слагаемые с $E_z$, $E_z'$, $E_z''$, что даёт неполное (и потенциально ошибочное) дисперсионное уравнение.

Аналогичная проблема для TM-моды: не ставится $H_z = 0$.

**Для изотропной среды** ($\varepsilon_{xx} = \varepsilon_{yy} = \varepsilon_{zz}$, $\mu_{xx} = \mu_{yy} = \mu_{zz}$) связи между $E_z$ и $H_z$ нет, и результат правильный. Но для анизотропного случая — нет.

### Ошибка 6 (мелкая). Дублирование вывода (Cell 44)

```python
show(lhs_ccE)
show(lhs_ccE)   # ← опечатка: второй show должен быть show(rhs_ccE)
```

---

## Резюме по степени серьёзности

| # | Описание | Серьёзность |
|---|---------|-------------|
| 1 | Неправильные формулы $k_y$ в markdown | Средняя (документация, но не код расчёта) |
| 2 | `ky_TE_cos` пропускает чётные моды | Средняя (влияет на верификацию, не на вывод $K_x^2$) |
| 3 | $e^{ik_y y}$ вместо $\sin/\cos$ | Низкая (для дисперсионного уравнения эквивалентно) |
| 4 | Скаляр `mu_xx` вместо тензора в коррекции | Низкая сейчас (коррекция = 0), высокая при модификации |
| 5 | **Нет $E_z=0$/$H_z=0$ — моды не разделены** | **Высокая** для анизотропного случая |
| 6 | Опечатка в show | Косметическая |

**Главный вывод**: для **изотропной** среды notebook даёт правильные дисперсионные уравнения. Для **анизотропной** — результат может быть неполным из-за связи TE и TM мод, которая в коде не обрабатывается.

You're out of extra usage ∙ Your limit resets Friday at 8:00 PM

      Продолжение рассмотрения инженерного алгоритма, используемого в численном  
		электродинамическом расчёте волнового двигателя. 
		Основная тестовая и расчётная  среда — файл MenDrive_real(3).ipynb.	
		Исследование будет проводиться в несколько этапов. 
		Этап 1.
		Проанализируй критически файл с выводом дисперсионного уравнения 
		MenDrive_wave_equation.ipynb
		На базе этого файла проверь правильность кода в ячейках
		"execution_count": 126, "execution_count": 127, "execution_count": 128
		файла MenDrive_real(3).ipynb.
		
Алгебраическая проверка подтверждена. Теперь составлю полный анализ.

---

## Этап 1. Критический анализ волнового уравнения и ячеек 126–128

### 1. Правильность формул K²_x,TE и K²_x,TM (вывод из первых принципов)

Исходя из уравнений Максвелла в СГС для анизотропной среды с диагональными тензорами ε = diag(ε_xx, ε_yy, ε_zz) и μ = diag(μ_xx, μ_yy, μ_zz), при полевом анзаце:

$$\vec{E}, \vec{H} \;\sim\; e^{i(k_z z - \omega t)} \cdot e^{i k_y y}$$

т.е. $\partial/\partial z \to ik_z$, $\partial/\partial y \to ik_y$.

**Аналитический вывод для TE-моды (E_z = 0):**

Из компонентных уравнений Максвелла поперечные компоненты E_x, E_y выражаются через H_z и его производную. Подстановка в уравнение для $\partial E_y/\partial x - ik_y E_x = \frac{i\omega}{c}\mu_{zz} H_z$ даёт волновое уравнение:

$$\frac{d^2 H_z}{dx^2} + K^2_{x,\text{TE}} \cdot H_z = 0$$

с точным результатом:

$$\boxed{K^2_{x,\text{TE}} = -\frac{k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{yy}\mu_{xx}}{\mu_{xx}} \left[\mu_{zz} + \frac{k_{y,H}^2\,\mu_{yy}}{k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{xx}\mu_{yy}}\right]}$$

Обозначая $D_{\text{TE},x} = k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{xx}\mu_{yy}$ и $D_{\text{TE},y} = k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{yy}\mu_{xx}$:

$$K^2_{x,\text{TE}} = -\frac{(k_{y,H}^2\mu_{yy} + \mu_{zz}\,D_{\text{TE},x}) \cdot D_{\text{TE},y}}{D_{\text{TE},x} \cdot \mu_{xx}}$$

Аналогично по принципу двойственности (ε ↔ μ, E ↔ H):

$$\boxed{K^2_{x,\text{TM}} = -\frac{(\varepsilon_{yy}k_{y,E}^2 + \varepsilon_{zz}\,D_{\text{TM},x}) \cdot D_{\text{TM},y}}{D_{\text{TM},x} \cdot \varepsilon_{xx}}}$$

где $D_{\text{TM},x} = k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{yy}\mu_{xx}$, $D_{\text{TM},y} = k_z^2 - \frac{\omega^2}{c^2}\varepsilon_{xx}\mu_{yy}$.

---

### 2. Проверка ячейки 126 ✅

```python
K2_x_TE = -(c^2*k_y_H^2*mu_yy + c^2*k_z^2*mu_zz - epsilon_xx*mu_yy*mu_zz*omega^2) \
          * (c^2*k_z^2 - epsilon_yy*mu_xx*omega^2) \
          /((c^2*k_z^2 - epsilon_xx*mu_yy*omega^2)*c^2*mu_xx)
```

Числитель первого множителя раскладывается:
$$c^2 k_{y,H}^2\mu_{yy} + c^2 k_z^2\mu_{zz} - \varepsilon_{xx}\mu_{yy}\mu_{zz}\omega^2 = c^2\bigl[k_{y,H}^2\mu_{yy} + \mu_{zz}\underbrace{(k_z^2 - \tfrac{\omega^2}{c^2}\varepsilon_{xx}\mu_{yy})}_{D_{\text{TE},x}}\bigr]$$

После деления на $c^2$ в знаменателе — **полностью совпадает** с аналитическим результатом. ✅

Формула K²_x,TM в ячейке 126 также верна по аналогичной проверке. ✅

---

### 3. Проверка ячейки 127 ✅

```python
K2_x_TE = -(k_y_H^2*mu_yy + mu_zz*(k_z^2 - epsilon_xx*mu_yy*omega^2/c^2)) \
          * (k_z^2 - epsilon_yy*mu_xx*omega^2/c^2) \
          /((k_z^2 - epsilon_xx*mu_yy*omega^2/c^2)*mu_xx)
```

Это ячейка 126 после сокращения $c^2$ — **алгебраически тождественна** ячейке 126. ✅ Обе формулы, TE и TM, верны.

---

### 4. Проверка ячейки 128 ❌ — ОШИБКА

```python
K2_x_TE = -(k_y_H^2*mu_yy/((k_z^2 - epsilon_xx*mu_yy*omega^2/c^2)) + mu_zz*(k_z^2 - epsilon_xx*mu_yy*omega^2/c^2)) \
          * (k_z^2 - epsilon_yy*mu_xx*omega^2/c^2) / (mu_xx)
```

Видна попытка вынести $1/D_{\text{TE},x}$ из знаменателя внутрь скобки. Но это сделано **неверно**: первый член $k_{y,H}^2\mu_{yy}$ корректно делится на $D_{\text{TE},x}$, однако второй член $\mu_{zz}$ ошибочно **умножается** на $D_{\text{TE},x}$ вместо того, чтобы остаться просто $\mu_{zz}$.

| Член | Ячейка 127 (верно) | Ячейка 128 (написано) | Ячейка 128 (должно быть) |
|------|--------------------|-----------------------|--------------------------|
| Первое слагаемое в скобке | $\dfrac{k_{y,H}^2\mu_{yy}}{D_{\text{TE},x}} \cdot D_{\text{TE},x}$ | $\dfrac{k_{y,H}^2\mu_{yy}}{D_{\text{TE},x}}$ ✅ | $\dfrac{k_{y,H}^2\mu_{yy}}{D_{\text{TE},x}}$ |
| Второе слагаемое в скобке | $\mu_{zz} \cdot D_{\text{TE},x}$ | $\mu_{zz} \cdot D_{\text{TE},x}$ ❌ | $\mu_{zz}$ |

Ошибка подтверждена символьно (SymPy): разность $K^2_{TE,127} - K^2_{TE,128}$ равна $\frac{\mu_{zz}(c^2k_z^2 - \varepsilon_{yy}\mu_{xx}\omega^2)(c^2k_z^2 - c^2 - \varepsilon_{xx}\mu_{yy}\omega^2)}{c^4\mu_{xx}} \neq 0$.

**Аналогичная ошибка в K²_x,TM ячейки 128** — ε_zz·D_TM,x вместо просто ε_zz.

---

### 5. Исправленная ячейка 128

```python
# Правильная ячейка 128: K²_x = -(kᵧ²·μ/D + μ_zz) · D_y / μ_xx
K2_x_TE = -(k_y_H**2*mu_yy/(k_z**2 - epsilon_xx*mu_yy*omega**2/c**2) + mu_zz) \
          * (k_z**2 - epsilon_yy*mu_xx*omega**2/c**2) / mu_xx

K2_x_TM = -(epsilon_yy*k_y_E**2/(k_z**2 - epsilon_yy*mu_xx*omega**2/c**2) + epsilon_zz) \
          * (k_z**2 - epsilon_xx*mu_yy*omega**2/c**2) / epsilon_xx
```

---

### 6. Дополнительные замечания по MenDrive_wave_equation.ipynb

Одна содержательная несоответствие: в **markdown** ячейки 1 для TM-моды указано $k_y = \frac{m\pi}{2b}$, тогда как нулевое граничное условие $E_z|_{y=\pm b} = 0$ для $E_z \propto \cos(k_y y)$ требует $\cos(k_y b) = 0$, т.е. $k_y = \frac{(2m+1)\pi}{2b}$ — это и реализовано в коде ячейки 3 (`ky_TM_cos = (1+2*m)*pi/(2*b)`). Текст markdown содержит опечатку, код верен.

---

**Итог этапа 1:**
- Ячейки **126** и **127** — корректны и эквивалентны друг другу.
- Ячейка **128** содержит **алгебраическую ошибку** в преобразовании: μ_zz (и ε_zz для TM) ошибочно умножены на D_TE,x (D_TM,x) вместо того чтобы остаться без изменений. Исправленные формулы приведены выше.




You're out of extra usage ∙ Your limit resets Friday at 8:00 PM